Load data

In [ ]:
!pip install huggingface_hub
!pip install transformers datasets tokenizers seqeval -q


Get data

In [ ]:
from utilities import preprocess
# get data
model_name="xlm-roberta-large"
language_code="ru"
data=preprocess(language_code=language_code, model_name=model_name, train=True)
tokenized_datasets, label_list, label2id, id2label, tokenizer= data
print(tokenized_datasets["train"][0])

c:\Users\user\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
c:\Users\user\anaconda3\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Map:   0%|          | 0/7560 [00:00<?, ? examples/s]

Map:   0%|          | 0/5152 [00:00<?, ? examples/s]

{'tokens': ['В', 'Пакистан', 'протестовать', 'против', 'отмена', 'приговор', 'за', 'богохульство', '.'], 'ner_tags': [10, 1, 10, 10, 10, 10, 10, 10, 10], 'input_ids': [0, 417, 174222, 34800, 27224, 4988, 183, 27145, 151609, 61, 85063, 244, 29524, 3280, 6, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 

In [ ]:
from transformers import AutoTokenizer,AutoModelForTokenClassification,AutoModelForTokenClassification, AutoConfig
from utilities import preprocess
from transformers import TrainingArguments, Trainer, IntervalStrategy
from transformers import DataCollatorForTokenClassification
from transformers import pipeline
from huggingface_hub import HfApi
import os


In [ ]:
def compute_metrics(eval_preds):
    """
    Function to compute the evaluation metrics for Named Entity Recognition (NER) tasks.
    The function computes precision, recall, F1 score and accuracy.

    Parameters:
    eval_preds (tuple): A tuple containing the predicted logits and the true labels.

    Returns:
    A dictionary containing the precision, recall, F1 score and accuracy.
    """
    pred_logits, labels = eval_preds

    pred_logits = np.argmax(pred_logits, axis=2)
    # the logits and the probabilities are in the same order,
    # so we don’t need to apply the softmax

    # We remove all the values where the label is -100
    predictions = [
        [label_list[eval_preds] for (eval_preds, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(pred_logits, labels)
    ]

    true_labels = [
      [label_list[l] for (eval_preds, l) in zip(prediction, label) if l != -100]
       for prediction, label in zip(pred_logits, labels)
   ]
    results = metric.compute(predictions=predictions, references=true_labels)
    return {
   "precision": results["overall_precision"],
   "recall": results["overall_recall"],
   "f1": results["overall_f1"],
  "accuracy": results["overall_accuracy"],
  }

In [ ]:


#def build_model(out_dir="test-ner", learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16, num_train_epochs=1, weight_decay=0.01):
def build_model(model_parameters):
    model_name=model_parameters["model_name"]
    tokenized_datasets, label_list, label2id, id2label, tokenizer= preprocess(language_code=model_parameters["language_code"], model_name=model_name, train=model_parameters["train"])
    config = AutoConfig.from_pretrained(model_name, num_labels=len(label_list) , id2label=id2label, label2id=label2id)
    model = AutoModelForTokenClassification.from_config(config)
    data_collator = DataCollatorForTokenClassification(tokenizer)
    args = TrainingArguments(
        output_dir=model_parameters["out_dir"],
        learning_rate=model_parameters["learning_rate"],
        per_device_train_batch_size=model_parameters["per_device_train_batch_size"],
        per_device_eval_batch_size=model_parameters["per_device_eval_batch_size"],
        num_train_epochs=model_parameters["num_train_epochs"],
        weight_decay=model_parameters["weight_decay"],
        #evaluation_strategy="epoch"
    )

    trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer
    compute_metrics=compute_metrics
    )
    
    return trainer

In [ ]:

model_parameters= {
    "language_code": language_code,
    "model_name":model_name,
    "train": True,
    "out_dir": language_code+"-ner",
    "learning_rate":2e-5,
    "per_device_train_batch_size":16,
    "per_device_eval_batch_size":16,
    "num_train_epochs":1,
    "weight_decay":0.01
}


trainer=build_model(model_parameters)
trainer.train()

: 

In [ ]:
from transformers import TrainingArguments, Trainer

def train_model(model_parameters):
    tokenized_datasets, label_list, label2id, id2label, tokenizer = preprocess(
        language_code=model_parameters["language_code"],
        model_name=model_parameters["model_name"],
        train=model_parameters["train"]
    )

    # Initialize model and data_collator here (you need to define or import them appropriately)
    model = get_model(model_parameters["model_name"], label2id, id2label)  # <-- make sure you define this
    data_collator = get_data_collator(tokenizer)  # <-- make sure you define this too

    args = TrainingArguments(
       

    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator
    )

    return trainer


Create model

In [ ]:
from transformers import AutoTokenizer,AutoModelForTokenClassification,AutoModelForTokenClassification, AutoConfig
from transformers import TrainingArguments, Trainer, IntervalStrategy
from transformers import DataCollatorForTokenClassification
from transformers import pipeline
from huggingface_hub import HfApi
import os

config = AutoConfig.from_pretrained(model_name, num_labels=len(label_list) , id2label=id2label, label2id=label2id)
model = AutoModelForTokenClassification.from_config(config)
data_collator = DataCollatorForTokenClassification(tokenizer)

args = TrainingArguments(
"test-ner",
#evaluation_strategy = "epoch",
learning_rate=2e-5,
per_device_train_batch_size=16,
per_device_eval_batch_size=16,
num_train_epochs=1,
weight_decay=0.01
)


In [ ]:
from utilities import preprocess
# get data
model_name="xlm-roberta-large"
data=preprocess(language_code='ru', model_name=model_name)
tokenized_datasets_ru, label_list, label2id, id2label, tokenizer= data[0], data[1],data[2], data[3], data[4]
print(tokenized_datasets_ru["train"][0])


Map:   0%|          | 0/7560 [00:00<?, ? examples/s]

Map:   0%|          | 0/5152 [00:00<?, ? examples/s]

Map:   0%|          | 0/10692 [00:00<?, ? examples/s]

{'tokens': ['В', 'Пакистан', 'протестовать', 'против', 'отмена', 'приговор', 'за', 'богохульство', '.'], 'ner_tags': [10, 1, 10, 10, 10, 10, 10, 10, 10], 'input_ids': [0, 417, 174222, 34800, 27224, 4988, 183, 27145, 151609, 61, 85063, 244, 29524, 3280, 6, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 

In [ ]:

# intitialize model
from transformers import AutoTokenizer,AutoModelForTokenClassification,AutoModelForTokenClassification, AutoConfig
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
from transformers import pipeline

config = AutoConfig.from_pretrained(model_name, num_labels=len(label_list) , id2label=id2label, label2id=label2id)
model = AutoModelForTokenClassification.from_config(config)
data_collator = DataCollatorForTokenClassification(tokenizer) 
args = TrainingArguments(
"test-ner",
evaluation_strategy = "epoch",
learning_rate=2e-5,
per_device_train_batch_size=16,
per_device_eval_batch_size=16,
num_train_epochs=3,
weight_decay=0.01,
)


C:\Users\maryz\AppData\Roaming\Python\Python311\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# train

trainer = Trainer(
    model,
    args,
   train_dataset=tokenized_datasets_ru["train"],
   eval_dataset=tokenized_datasets_ru["validation"],
   data_collator=data_collator,
   tokenizer=tokenizer
)
trainer.train()

: 

In [ ]:
import torch # Import torch library
from utilities import preprocess
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)

# --- Configuration ---
model_name = "xlm-roberta-large"
language_code = 'ru'
output_dir = "test-ner" # Define output directory for TrainingArguments
learning_rate = 2e-5
train_batch_size = 16
eval_batch_size = 16
num_train_epochs = 3
weight_decay = 0.01

# --- Check for GPU availability ---
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA not available. Using CPU.")

# --- Data Preprocessing ---
# Assuming 'utilities.preprocess' returns the necessary components
# Make sure the preprocess function is defined or imported correctly
try:
    data = preprocess(language_code=language_code, model_name=model_name)
    tokenized_datasets_ru, label_list, label2id, id2label, tokenizer = data[0], data[1], data[2], data[3], data[4]
    print("Sample from training data:")
    print(tokenized_datasets_ru["train"][0])
    num_labels = len(label_list)
except NameError:
    print("Error: The 'preprocess' function is not defined.")
    print("Please ensure 'utilities.py' is in the same directory or accessible in your Python path,")
    print("and that it contains the 'preprocess' function.")
    # Example placeholder data if preprocess fails - replace with actual loading if needed
    tokenized_datasets_ru = None # Set to None or load dummy data
    label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"] # Example labels
    label2id = {label: i for i, label in enumerate(label_list)}
    id2label = {i: label for i, label in enumerate(label_list)}
    num_labels = len(label_list)
    tokenizer = AutoTokenizer.from_pretrained(model_name) # Load tokenizer separately if needed
    print("\nWARNING: Using placeholder data because 'preprocess' failed.")


# --- Initialize Model ---
print(f"\nInitializing model: {model_name}")
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
# Load the model
model = AutoModelForTokenClassification.from_pretrained(model_name, config=config)

# --- Move Model to GPU (if available) ---
# Explicitly move the model to the selected device (GPU or CPU)
model.to(device)
print(f"Model moved to device: {device}")

# --- Data Collator ---
# Handles padding and batch preparation.
# The Trainer will automatically move data batches to the correct device.
data_collator = DataCollatorForTokenClassification(tokenizer)

# --- Training Arguments ---
# Configure training parameters.
# `TrainingArguments` automatically detects and uses CUDA if available by default.
# Setting `no_cuda=False` (default) ensures GPU is used if possible.
args = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy="epoch",
    learning_rate=learning_rate,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    num_train_epochs=num_train_epochs,
    weight_decay=weight_decay,
    logging_dir='./logs', # Optional: directory for logs
    logging_steps=10,     # Optional: log metrics every 10 steps
    save_strategy="epoch", # Optional: save model checkpoint every epoch
    load_best_model_at_end=True, # Optional: load the best model found during training
    # no_cuda=False # Default is False, explicitly setting it ensures GPU usage if available
)

# --- Initialize Trainer ---
# The Trainer class handles the training loop, evaluation, and device placement.
if tokenized_datasets_ru: # Check if data was loaded successfully
    trainer = Trainer(
        model=model, # The model is already on the correct device
        args=args,
        train_dataset=tokenized_datasets_ru["train"],
        eval_dataset=tokenized_datasets_ru["validation"],
        data_collator=data_collator,
        tokenizer=tokenizer
    )

    # --- Train the Model ---
    print("\nStarting training...")
    trainer.train()
    print("Training finished.")

    # --- Save the final model ---
    trainer.save_model(f"{output_dir}/final_model")
    tokenizer.save_pretrained(f"{output_dir}/final_model")
    print(f"Model saved to {output_dir}/final_model")

    # --- Example of using the trained model (optional) ---
    print("\nExample prediction using the trained model:")
    # Load the fine-tuned model and tokenizer
    # The pipeline automatically handles moving data to the model's device
    ner_pipeline = pipeline(
        "token-classification",
        model=f"{output_dir}/final_model",
        tokenizer=f"{output_dir}/final_model",
        # device=0 if torch.cuda.is_available() else -1 # Explicitly set device for pipeline if needed
    )

    example_text = "Пример текста на русском языке для проверки NER." # Example text in Russian
    results = ner_pipeline(example_text)
    print(f"Input: {example_text}")
    print(f"Predictions: {results}")

else:
    print("\nSkipping training because data preprocessing failed.")



: 

In [ ]:
args = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy="steps",
    eval_steps=500,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    logging_dir='./logs',
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True
)


In [ ]:


# save prediction
predictions=trainer.predict(tokenized_datasets_ru["test"])
final_predictions_test = [
    [id2label[p] for p in sentence] for sentence in predictions
]

iob_predictions=[]
for sent_idx, final_pred in enumerate(final_predictions_test):

    tokens = tokenized_datasets_ru["test"]["tokens"][sent_idx]  # Tokens from your dataset
    tokens_ids = tokenized_datasets_ru["test"]["input_ids"][sent_idx]  # Tokens from your dataset
    sentence_iob = []
    pred_label_cleaned=final_pred[1:len(tokens)+1]
    for token_idx, token in enumerate(tokens):
        pred_label = pred_label_cleaned[token_idx]  # Get the label from final_predictions
        sentence_iob.append(f"{token_idx+1}\t{tokens[token_idx]}\t{pred_label}")

    iob_predictions.append(sentence_iob)

with open("ner_predictions.iob", "w") as f:
    for sentence in iob_predictions:
        for line in sentence:
            f.write(f"{line}\n")
        f.write("\n")  # Separate sentences with a blank line
#save models

from huggingface_hub import HfApi

trainer.save_model("./my_ner_model")
tokenizer.save_pretrained("./my_ner_model")
# Load your model and tokenizer
model = AutoModelForTokenClassification.from_pretrained("./my_ner_model")
tokenizer = AutoTokenizer.from_pretrained("./my_ner_model")

# Push to the hub
model.push_to_hub("your-username/your-model-name")
tokenizer.push_to_hub("your-username/your-model-name")
